In [2]:
# Set up cell
# Imports and Installs
import psycopg2
import requests
import os
import logging

# Setup Logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)
logger.info("logging at absolute path")

# Function for fetching api data
def fetch_api(url, key):
    try:
        r = requests.get(url, auth=(key, ''))
        r.raise_for_status() # Stops code if status is 4xx/5xx
        return r.json()
    except Exception as e:
        logger.error(f"Failed {url}: {e}")
        return None
    
# Function for pushing data to database
def push_to_db(table_name, columns, values):
    db_url = os.getenv('DATABASE_URL')
    try: 
        with psycopg2.connect(db_url) as conn:
            with conn.cursor() as cur:
                # Joining the column names into a single string for SQL processing
                str_col= ', '.join(columns)
                # Multipling the number of varibales by the number of values in order to process the correct number of values
                placeholders = ', '.join(['%s'] * len(values))
                query = f"""INSERT INTO {table_name} ({str_col})
                VALUES ({placeholders})
                ON CONFLICT (valid_from)
                DO NOTHING
                """

                cur.execute(query, values)
                conn.commit()
            logger.info(f"Successfully pushed data to {table_name}.")
    except psycopg2.Error as e:
        logger.error(f"Database error: {e}")

2026-05-21 14:37:30,196 - INFO - logging at absolute path


In [3]:
def lambda_handler(event, context):
    logger.info("Pipeline triggered")
    api_key = os.getenv('API_KEY')

    # 1. CONSUMPTION
    mpan = os.getenv('MPAN')
    serial = os.getenv('SERIAL_NUM')
    url_usage = f"https://api.octopus.energy/v1/electricity-meter-points/{mpan}/meters/{serial}/consumption/"
    data_usage = fetch_api(url_usage, api_key)

    if data_usage: 
        for usage in data_usage['results']:
            consumption = usage.get('consumption')
            valid_from = usage.get('interval_start')
            valid_to = usage.get('interval_end')
            if consumption is not None and valid_from is not None:
                consumption_cols = ('consumption_kwh', 'valid_from', 'valid_to')
                consumption_vals = (consumption, valid_from, valid_to)
                push_to_db('consumption', consumption_cols, consumption_vals)
            else:
                logger.warning(f"Skipping incomplete record: {usage}")
                
    # 2. ACCOUNT & TARIFF DATA
    acc_num = os.getenv('ACC_NUM')
    url_acc = f"https://api.octopus.energy/v1/accounts/{acc_num}/"
    data_acc = fetch_api(url_acc, api_key)
    
    # Define old and new codes in pairs
    codes_to_fetch = [
        {
            'tariff': 'E-1R-BULB-S2PAYG-VAR-V1-C',
            'product': 'BULB-S2PAYG-VAR-V1'
        },
        {
            'tariff': 'E-1R-SMART-PREPAY-VAR-22-10-22-C',
            'product': 'SMART-PREPAY-VAR-22-10-22'
        }
    ]
    # Loop through the list
    for item in codes_to_fetch:
        t_code = item['tariff']
        p_code = item['product']

        # 3. UNIT RATES
        url_rate = f"https://api.octopus.energy/v1/products/{p_code}/electricity-tariffs/{t_code}/standard-unit-rates/"
        data_rate = fetch_api(url_rate, api_key)
        
        if data_rate:
            for rate in data_rate['results']:
                valid_from = rate.get('valid_from')
                valid_to = rate.get('valid_to')
                value_exc_vat = rate.get('value_exc_vat')
                value_inc_vat = rate.get('value_inc_vat')

                rate_cols = ('valid_from', 'valid_to', 'cost_exc_vat', 'cost_inc_vat')
                rate_vals = (valid_from, valid_to, value_exc_vat, value_inc_vat)

                print(f"DEBUG: Tariff: {t_code} | Date: {valid_from} | Rate: {value_inc_vat}")
                push_to_db('unit_rates', rate_cols, rate_vals)

        # 4. STANDING CHARGES
        url_standing = f"https://api.octopus.energy/v1/products/{p_code}/electricity-tariffs/{t_code}/standing-charges/"
        data_standing = fetch_api(url_standing, api_key)
        
        if data_standing:
            for standing in data_standing['results']:
                valid_from = standing.get('valid_from')
                valid_to = standing.get('valid_to')
                value_exc_vat = standing.get('value_exc_vat')
                value_inc_vat = standing.get('value_inc_vat')

                standing_cols = ('valid_from', 'valid_to', 'cost_exc_vat', 'cost_inc_vat')
                standing_vals = (valid_from, valid_to, value_exc_vat, value_inc_vat)

                print(f"DEBUG: Tariff: {t_code} | Date: {valid_from} | Standing: {value_inc_vat}")
                push_to_db('standing_charges', standing_cols, standing_vals)

    return {'statusCode': 200, 'body': 'Success'}